### Multi-GPU layout
Stage 1: Faster R-CNN uses 2-process NCCL DistributedDataParallel.

Stage 2: U-Net uses 2-process NCCL DistributedDataParallel. Crop data are stored as
disk-backed NumPy memory maps, and the U-Net DataLoader uses `num_workers=0` and
`pin_memory=False` to keep Kaggle host RAM stable.

Final inference/export reconstructs ordinary single-model copies from the saved state dictionaries.

In [ ]:
%pip install -q segmentation-models-pytorch

import os, re, csv, sys, json, math, time, random, warnings, collections, gc, gc
from pathlib import Path
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")

CFG = dict(
    seed=42, val_frac=0.10, test_frac=0.10,
    crop=256, pad=0.15,   # widened from 0.05: more context around each tooth so boundary caries isn't clipped
    # stage 1: tooth detector
    det_epochs=60, det_bs=2, det_lr=0.005, det_patience=12, det_budget_h=2.5,
    # stage 2: caries U-Net
    seg_epochs=150, seg_bs=16, seg_lr=3e-4, seg_patience=25, seg_budget_h=3.5,
    # --- false-positive reduction (Stage 2) ---
    seg_neg_ratio=1.0,          # caries-free crops per caries crop per epoch. Capped by availability (~11.3k neg vs ~8.0k pos => max ~1.4);
                                # mined hard negatives REPLACE part of this quota, so raising it only helps if you add more negatives
    seg_tv_alpha=0.5, seg_tv_beta=0.5,   # Tversky FP / FN weights (was 0.3 / 0.7 = recall-biased)
    seg_fp_lambda=0.5,          # extra penalty on the strongest activation inside caries-free crops
    seg_hn_start=5, seg_hn_every=5, seg_hn_frac=0.5, seg_hn_min_px=30,   # hard-negative mining
    seg_sel_fp_weight=0.5,      # checkpoint score = val Dice - weight * (fraction of empty val crops with a FP blob)
    seg_extra_val_imgs=40, seg_extra_val_tiles=300,   # held-out pseudo-labeled panoramics / pre-cropped tiles -> larger Stage-2 val set
    pp_own_box_margin=0.04,     # each crop only votes inside its own tooth box (+ this margin); overlaps are averaged
    hard_stop_h=10.0,
    seg_reuse_ckpt=False,     # wall-clock cap for the whole notebook (Kaggle kills sessions at 12 h)
    reuse_ckpt=True,     # True = skip a stage if its *_best.pt is already in /kaggle/working (crash recovery, same session only)
)
OUT = Path("/kaggle/working")
T0 = time.time()
def elapsed_h(): return (time.time() - T0) / 3600
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def make_scaler():
    """Create a CUDA GradScaler across PyTorch versions used by Kaggle."""
    try:
        return torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

seed_all(CFG["seed"])

assert torch.cuda.is_available(), "No GPU -> Kaggle Settings -> Accelerator -> GPU"
N_GPUS = torch.cuda.device_count()
assert N_GPUS >= 2, f"This notebook is configured for 2 GPUs, but Kaggle reports {N_GPUS} GPU(s). Select a 2xT4/P100 GPU accelerator."
GPU_IDS = list(range(min(2, N_GPUS)))
dev = torch.device(f"cuda:{GPU_IDS[0]}")
torch.backends.cudnn.benchmark = True
print("PyTorch:", torch.__version__)
for _gid in GPU_IDS:
    print(f"GPU {_gid}:", torch.cuda.get_device_name(_gid), torch.cuda.get_device_capability(_gid))
print(f"Using GPUs: {GPU_IDS}")
HIST = dict(det_loss=[], det_f1=[], seg_loss=[], seg_dice=[])


In [ ]:
%%writefile dental_inference.py
"""
dental_inference.py  --  Dual-stage dental caries segmentation (inference + model bundle I/O).

Stage 1: Faster R-CNN (MobileNetV3-Large FPN) localises individual teeth in a panoramic radiograph.
Stage 2: U-Net (EfficientNet-B0 encoder) segments carious pixels inside each tooth crop.

Usage (outside Kaggle):
    from dental_inference import DualStagePredictor
    pred = DualStagePredictor.from_pickle("dental_caries_dualstage.pkl")
    out = pred.predict("some_panoramic.png")      # dict: mask, prob, boxes, scores
    cv2.imwrite("mask.png", out["mask"] * 255)

Requires: torch, torchvision, segmentation-models-pytorch, opencv-python, numpy.
"""
import os
import pickle
import warnings

import cv2
import numpy as np
import torch

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


# ----------------------------------------------------------------------------- cropping utilities
def expand_box(box, W, H, pad=0.05):
    """Expand an (x1,y1,x2,y2) box by `pad` (fraction of its size) and clip to the image. Returns ints."""
    x1, y1, x2, y2 = [float(v) for v in box]
    w, h = x2 - x1, y2 - y1
    x1 = max(0, int(np.floor(x1 - pad * w)))
    y1 = max(0, int(np.floor(y1 - pad * h)))
    x2 = min(W, int(np.ceil(x2 + pad * w)))
    y2 = min(H, int(np.ceil(y2 + pad * h)))
    return x1, y1, x2, y2


def crop_tooth(gray, box, size=256, pad=0.05):
    """Crop one tooth from a uint8 grayscale panoramic image, apply CLAHE, resize to size x size.
    Returns (crop_uint8 [size,size], padded_box) or (None, None) if the box is degenerate."""
    H, W = gray.shape[:2]
    x1, y1, x2, y2 = expand_box(box, W, H, pad)
    if (x2 - x1) < 8 or (y2 - y1) < 8:
        return None, None
    c = gray[y1:y2, x1:x2]
    c = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(c)
    interp = cv2.INTER_AREA if min(c.shape[:2]) > size else cv2.INTER_LINEAR
    c = cv2.resize(c, (size, size), interpolation=interp)
    return c, (x1, y1, x2, y2)


def crop_mask(mask01, padded_box, size=256):
    """Crop + resize a binary {0,1} uint8 mask with the same padded box used for the image crop."""
    x1, y1, x2, y2 = padded_box
    m = cv2.resize(mask01[y1:y2, x1:x2] * 255, (size, size), interpolation=cv2.INTER_AREA)
    return (m > 127).astype(np.uint8)


def prep(xb):
    """(B,1,S,S) float in [0,1] -> (B,3,S,S) ImageNet-normalised (gray replicated to 3 channels)."""
    mean = torch.tensor(IMAGENET_MEAN, device=xb.device, dtype=xb.dtype).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=xb.device, dtype=xb.dtype).view(1, 3, 1, 1)
    return (xb.repeat(1, 3, 1, 1) - mean) / std


def postprocess_prob(prob, lo=0.5, hi=None, min_area=0):
    """Probability map -> binary mask with false-positive suppression.
    * hysteresis: pixels > `lo` form candidate blobs; a blob survives only if its peak probability >= `hi`
      (hi=None/<=lo disables the peak requirement),
    * blobs smaller than `min_area` pixels (original image resolution) are dropped."""
    low = (prob > lo).astype(np.uint8)
    if not low.any():
        return low
    n, lab, stats, _ = cv2.connectedComponentsWithStats(low, connectivity=8)
    if n <= 1:
        return low
    keep = stats[:, cv2.CC_STAT_AREA] >= min_area
    if hi is not None and hi > lo:
        ys, xs = np.nonzero(low)
        peak = np.zeros(n, np.float32)
        np.maximum.at(peak, lab[ys, xs], prob[ys, xs])
        keep &= peak >= hi
    keep[0] = False
    return keep[lab].astype(np.uint8)


# ----------------------------------------------------------------------------- model builders
def build_unet(encoder="efficientnet-b0", pretrained=True, verbose=True):
    """U-Net with an EfficientNet-B0 encoder (via segmentation_models_pytorch).
    Tries several equivalent encoder names because they differ between smp versions.
    Returns (model, encoder_name_used, got_pretrained_weights)."""
    import segmentation_models_pytorch as smp

    names = [encoder] + [n for n in ("efficientnet-b0", "timm-efficientnet-b0", "tu-efficientnet_b0") if n != encoder]
    weight_options = (["imagenet"] if pretrained else []) + [None]
    last_err = None
    for w in weight_options:
        for name in names:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    m = smp.Unet(encoder_name=name, encoder_weights=w, in_channels=3, classes=1)
                if verbose:
                    print(f"[build_unet] encoder={name} weights={w}")
                    if pretrained and w is None:
                        print("  !! WARNING: ImageNet weights could NOT be loaded (is Internet ON?). "
                              "Training from scratch -> expect worse results.")
                return m, name, (w == "imagenet")
            except Exception as e:  # noqa
                last_err = e
    raise RuntimeError(f"Could not build U-Net: {last_err}")


def build_detector(pretrained=True, min_size=800, max_size=1700, verbose=True):
    """Faster R-CNN with MobileNetV3-Large FPN backbone, 2 classes (background + tooth).
    Returns (model, got_pretrained_backbone)."""
    from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn

    kw = dict(num_classes=2, min_size=min_size, max_size=max_size, box_detections_per_img=64)
    if pretrained:
        try:
            from torchvision.models import MobileNet_V3_Large_Weights
            m = fasterrcnn_mobilenet_v3_large_fpn(
                weights=None, weights_backbone=MobileNet_V3_Large_Weights.IMAGENET1K_V1, **kw)
            if verbose:
                print("[build_detector] MobileNetV3-Large FPN, ImageNet backbone weights loaded")
            return m, True
        except Exception as e:  # noqa
            print(f"  !! WARNING: could not load ImageNet backbone weights ({type(e).__name__}). "
                  "Is Internet ON? Training the detector from scratch.")
    m = fasterrcnn_mobilenet_v3_large_fpn(weights=None, weights_backbone=None, **kw)
    return m, False


# ----------------------------------------------------------------------------- predictor
class DualStagePredictor:
    """Detector -> per-tooth crops -> U-Net -> stitched panoramic mask."""

    def __init__(self, unet, detector, size=256, pad=0.05, det_thr=0.5, unet_thr=0.5,
                 unet_encoder="efficientnet-b0", det_min_size=800, det_max_size=1700, device=None,
                 unet_hi=None, min_area=0, own_box_margin=0.04, merge="weighted"):
        self.device = torch.device(device) if device else next(unet.parameters()).device
        self.unet = unet.to(self.device).eval()
        self.det = detector.to(self.device).eval()
        self.size, self.pad = size, pad
        self.det_thr, self.unet_thr = det_thr, unet_thr
        # false-positive suppression: unet_thr = low (grow) threshold, unet_hi = required blob peak,
        # min_area = smallest blob kept (px), merge = 'weighted' (own-box vote, averaged) or 'max' (legacy)
        self.unet_hi, self.min_area = unet_hi, min_area
        self.own_box_margin, self.merge = own_box_margin, merge
        self.unet_encoder = unet_encoder
        self.det_min_size, self.det_max_size = det_min_size, det_max_size

    # -- helpers
    @staticmethod
    def _load_gray(image):
        if isinstance(image, (str, os.PathLike)):
            g = cv2.imread(str(image), cv2.IMREAD_GRAYSCALE)
            if g is None:
                raise IOError(f"cannot read image: {image}")
            return g
        a = np.asarray(image)
        if a.ndim == 3:
            a = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
        if a.dtype != np.uint8:
            a = np.clip(a, 0, 255).astype(np.uint8)
        return a

    @property
    def _amp(self):
        return self.device.type == "cuda"

    # -- stage 1
    @torch.no_grad()
    def detect(self, gray, thr=None):
        thr = self.det_thr if thr is None else thr
        x = torch.from_numpy(gray).float().div(255).unsqueeze(0).repeat(3, 1, 1).to(self.device)
        o = self.det([x])[0]
        k = o["scores"] >= thr
        return o["boxes"][k].float().cpu().numpy(), o["scores"][k].float().cpu().numpy()

    # -- stage 2
    @torch.no_grad()
    def segment_crops(self, crops_u8, bs=32, tta_flip=True):
        """crops_u8: (N,S,S) uint8 -> (N,S,S) float32 probabilities.
        tta_flip=True averages each crop's prediction with its horizontally-flipped version
        (test-time augmentation): ~2x U-Net compute, no retraining needed, usually a small free Dice gain."""
        out = []
        for i in range(0, len(crops_u8), bs):
            xb = torch.from_numpy(np.ascontiguousarray(crops_u8[i:i + bs])).float().div(255).unsqueeze(1).to(self.device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self._amp):
                lg = self.unet(prep(xb)).float()
                if tta_flip:
                    lg_f = torch.flip(self.unet(prep(torch.flip(xb, dims=[-1]))).float(), dims=[-1])
                    lg = (lg + lg_f) / 2
            out.append(torch.sigmoid(lg).squeeze(1).cpu().numpy())
        return np.concatenate(out) if out else np.zeros((0, self.size, self.size), np.float32)

    # -- full pipeline
    def _own_region(self, box, pbox, W, H):
        """0/1 weight over the padded crop window: 1 only inside this tooth's own box (+ margin)."""
        px1, py1, px2, py2 = pbox
        ox1, oy1, ox2, oy2 = expand_box(box, W, H, self.own_box_margin)
        w = np.zeros((py2 - py1, px2 - px1), np.float32)
        w[max(oy1, py1) - py1:min(oy2, py2) - py1, max(ox1, px1) - px1:min(ox2, px2) - px1] = 1.0
        return w

    @torch.no_grad()
    def predict_prob(self, image, boxes=None):
        """Stages 1+2 without thresholding. Returns (prob float32 HxW, boxes (N,4), scores (N,))."""
        gray = self._load_gray(image)
        H, W = gray.shape
        if boxes is None:
            boxes, scores = self.detect(gray)
        else:
            boxes = np.asarray(boxes, np.float32).reshape(-1, 4)
            scores = np.ones(len(boxes), np.float32)
        crops, pboxes, oboxes = [], [], []
        for b in boxes:
            c, pb = crop_tooth(gray, b, self.size, self.pad)
            if c is not None:
                crops.append(c)
                pboxes.append(pb)
                oboxes.append(b)
        prob = np.zeros((H, W), np.float32)
        if crops:
            probs = self.segment_crops(np.stack(crops))
            if self.merge == "max":                      # legacy: any crop firing wins (padding context included)
                for p, (x1, y1, x2, y2) in zip(probs, pboxes):
                    pr = cv2.resize(p, (x2 - x1, y2 - y1), interpolation=cv2.INTER_LINEAR)
                    prob[y1:y2, x1:x2] = np.maximum(prob[y1:y2, x1:x2], pr)
            else:                                        # each crop votes only inside its own tooth box; votes averaged
                acc = np.zeros((H, W), np.float32)
                wsum = np.zeros((H, W), np.float32)
                for p, pb, ob in zip(probs, pboxes, oboxes):
                    x1, y1, x2, y2 = pb
                    pr = cv2.resize(p, (x2 - x1, y2 - y1), interpolation=cv2.INTER_LINEAR)
                    w = self._own_region(ob, pb, W, H)
                    acc[y1:y2, x1:x2] += pr * w
                    wsum[y1:y2, x1:x2] += w
                np.divide(acc, wsum, out=prob, where=wsum > 0)
        return prob, np.asarray(boxes), np.asarray(scores)

    @torch.no_grad()
    def predict(self, image, boxes=None):
        """image: path or ndarray. boxes: optional (N,4) xyxy tooth boxes (skips the detector, e.g. ground truth).
        Returns dict(mask uint8 {0,1} HxW, prob float32 HxW, boxes (N,4), scores (N,))."""
        prob, boxes, scores = self.predict_prob(image, boxes)
        mask = postprocess_prob(prob, self.unet_thr, self.unet_hi, self.min_area)
        return dict(mask=mask, prob=prob, boxes=boxes, scores=scores)

    # -- (de)serialisation: one pickle file holding everything needed to rebuild the pipeline
    def to_bundle(self, extra=None):
        cpu = lambda sd: {k: v.detach().cpu() for k, v in sd.items()}  # noqa: E731

        # The predictor normally receives the underlying detector, but keep export
        # robust if a DataParallel/wrapper object is passed in.
        det_model = self.det
        if hasattr(det_model, "module"):
            det_model = det_model.module
        if hasattr(det_model, "detector"):
            det_model = det_model.detector

        unet_model = self.unet
        if hasattr(unet_model, "module"):
            unet_model = unet_model.module

        return dict(
            format_version=2,
            unet=dict(encoder=self.unet_encoder, state_dict=cpu(unet_model.state_dict())),
            detector=dict(
                state_dict=cpu(det_model.state_dict()),
                min_size=self.det_min_size,
                max_size=self.det_max_size,
            ),
            settings=dict(
                size=self.size, pad=self.pad,
                det_thr=self.det_thr, unet_thr=self.unet_thr,
                unet_hi=self.unet_hi, min_area=self.min_area,
                own_box_margin=self.own_box_margin, merge=self.merge,
                mean=IMAGENET_MEAN, std=IMAGENET_STD,
            ),
            extra=extra or {},
        )

    def save_pickle(self, path, extra=None):
        with open(path, "wb") as f:
            pickle.dump(self.to_bundle(extra), f, protocol=4)
        return path

    @classmethod
    def from_bundle(cls, bundle, device=None):
        device = torch.device(device) if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        unet, enc, _ = build_unet(bundle["unet"]["encoder"], pretrained=False, verbose=False)
        unet.load_state_dict(bundle["unet"]["state_dict"])
        d = bundle["detector"]
        det, _ = build_detector(pretrained=False, min_size=d["min_size"], max_size=d["max_size"], verbose=False)
        det.load_state_dict(d["state_dict"])
        s = bundle["settings"]
        return cls(unet, det, size=s["size"], pad=s["pad"], det_thr=s["det_thr"], unet_thr=s["unet_thr"],
                   unet_encoder=enc, det_min_size=d["min_size"], det_max_size=d["max_size"], device=device,
                   unet_hi=s.get("unet_hi"), min_area=s.get("min_area", 0),
                   own_box_margin=s.get("own_box_margin", 0.04), merge=s.get("merge", "max"))  # v1 bundles -> legacy behaviour

    @classmethod
    def from_pickle(cls, path, device=None):
        with open(path, "rb") as f:
            return cls.from_bundle(pickle.load(f), device)


In [ ]:
import importlib, dental_inference as DI
importlib.reload(DI)
from dental_inference import DualStagePredictor, build_unet, build_detector, crop_tooth, crop_mask, prep, postprocess_prob, expand_box


## 1. Data: find images / caries masks / tooth boxes, and build the record list
The tooth-box annotation format is auto-detected and **validated against the caries masks** (boxes must actually cover the lesions).
If detection fails, the cell prints what it found, so you can paste it and we fix the parser in one shot.


In [4]:
import subprocess
# ------------------------------ Kaggle dataset paths ------------------------------
# thunderpede/panoramic-dental-dataset and testpeaky/dental are the same 100-image
# mirror, and it's the ONLY source with tooth-box annotations (bboxes_teeth/) - the
# base DC1000 release (fetched below) ships caries masks but no tooth boxes at all.
# So: ROOT (used for both the detector AND as the box-labeled part of Stage 2) always
# stays the box-annotated set; any bigger unboxed caries-segmentation folder found in
# the full release is exposed separately as EXTRA_IMG_DIR/EXTRA_LBL_DIR, to be
# pseudo-labeled with our own trained detector and folded into Stage-2 training only
# (see the new cell after the detector is trained).

IN = Path("/kaggle/input")
FULL_DC1000_GDRIVE_URL = "https://drive.google.com/file/d/1Xn1oGHvhGF9GbkcLEtCOV5QvWWqt1y62/view?usp=drive_link"
FULL_DC1000_BAIDU_URL = "https://pan.baidu.com/s/1jRXsSQIr8mm3EyGYELv9kg?pwd=rsoc"  # manual fallback, pwd: rsoc
FULL_DC1000_DIR = Path("/kaggle/working/dc1000_full")
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
LABEL_ALIASES = ("labels", "labels_clean", "masks", "mask")

def n_imgs(d):
    d = Path(d)
    if not d.is_dir():
        return 0
    return sum(1 for p in d.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)

def find_dataset_roots(base):
    """Every folder under `base` with an images/ subfolder plus a labels-like subfolder.
    Returns [(folder, label_subdir_name), ...]."""
    hits = []
    for dp, dn, fn in os.walk(base):
        low = {d.lower(): d for d in dn}
        if "images" not in low:
            continue
        for alias in LABEL_ALIASES:
            if alias in low:
                hits.append((Path(dp), low[alias]))
                break
    return hits

def find_annotations_dir(root, max_up=2, max_down=3):
    """Search a couple of levels around `root` for a folder literally named annotations/
    (it isn't always a sibling of images/+labels/ - e.g. shared one level up alongside
    train/, test/, org_train_dataset/, ...). Returns Path or None."""
    search_roots, p = [root], root
    for _ in range(max_up):
        p = p.parent
        search_roots.append(p)
    hits = []
    for sr in search_roots:
        for dp, dn, fn in os.walk(sr):
            depth = len(Path(dp).relative_to(sr).parts)
            if depth > max_down:
                dn[:] = []
                continue
            if Path(dp).name.lower() == "annotations":
                hits.append(Path(dp))
    if not hits:
        return None
    def dist(h):
        try: return os.path.relpath(h, root).count(os.sep)
        except ValueError: return 99
    return sorted(set(hits), key=lambda h: (dist(h), -n_imgs(h)))[0]

def classify(base):
    """[(root, label_subdir_name, annotations_dir_or_None, n_images), ...] for every
    images/+labels-like folder under `base`."""
    out = []
    for root, lbl in find_dataset_roots(base):
        out.append((root, lbl, find_annotations_dir(root), n_imgs(root / "images")))
    return out

boxed = [c for c in classify(IN) if c[2] is not None]
print("box-annotated (images+labels+annotations) candidates under /kaggle/input:",
      [(str(c[0]), c[3]) for c in boxed])
ROOT_INFO = max(boxed, key=lambda c: c[3]) if boxed else None

SMALL_MIRROR_SIZE = 200  # the testpeaky/thunderpede mirror only has 100 box-annotated images
if ROOT_INFO is None or ROOT_INFO[3] < SMALL_MIRROR_SIZE:
    have = ROOT_INFO[3] if ROOT_INFO else 0
    print(f"\nOnly {have} box-annotated images available as a Kaggle Input - fetching the full "
          f"DC1000 release from Google Drive to check for more (needs Internet ON)...")
    try:
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        FULL_DC1000_DIR.mkdir(parents=True, exist_ok=True)
        zpath = FULL_DC1000_DIR / "dc1000.zip"
        if not zpath.exists() or zpath.stat().st_size == 0:
            subprocess.run(["gdown", "--fuzzy", FULL_DC1000_GDRIVE_URL, "-O", str(zpath)],
                            check=True, timeout=1800)
        import zipfile
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(FULL_DC1000_DIR)
        dl_boxed = [c for c in classify(FULL_DC1000_DIR) if c[2] is not None]
        print("box-annotated candidates after download:", [(str(c[0]), c[3]) for c in dl_boxed])
        dl_best = max(dl_boxed, key=lambda c: c[3]) if dl_boxed else None
        if dl_best and dl_best[3] > have:
            ROOT_INFO = dl_best
        else:
            print("  The full release doesn't add a bigger box-annotated set (it only ships "
                  "unboxed caries-segmentation images - see EXTRA_* below). Keeping the "
                  "existing box-annotated Input for the detector and its share of Stage 2.")
    except Exception as e:
        print(f"  !! Google Drive download failed ({type(e).__name__}: {e}).")
        print(f"  Manual fallback: download {FULL_DC1000_GDRIVE_URL}")
        print(f"  (or Baidu: {FULL_DC1000_BAIDU_URL}), unzip, Add Data -> Upload as a new")
        print("  private Kaggle dataset, add it as an Input, then rerun this cell.")
        print("  Falling back to whatever box-annotated set was found under /kaggle/input for now.")

assert ROOT_INFO is not None, (
    f"No images/+labels/+annotations folder (the only kind with tooth boxes) found anywhere "
    f"under /kaggle/input. /kaggle/input contains: {[p.name for p in IN.iterdir()]}"
)
ROOT, LBL_NAME, ANN_DIR, _n = ROOT_INFO
IMG_DIR, LBL_DIR = ROOT / "images", ROOT / LBL_NAME
print(f"\nUsing box-annotated dataset root: {ROOT}  ({n_imgs(IMG_DIR)} images, "
      f"labels='{LBL_NAME}', annotations={ANN_DIR})")
assert IMG_DIR.is_dir(), f"Missing images folder: {IMG_DIR}"
assert LBL_DIR.is_dir(), f"Missing labels folder: {LBL_DIR}"
assert ANN_DIR is not None and ANN_DIR.is_dir(), f"Missing annotations folder near {ROOT}"

# ---- extra, unboxed caries-segmentation images (no tooth boxes) for later Stage-2-only
# augmentation. The full DC1000 release ships TWO structurally different extra pools, and
# they need different treatment - so both are discovered here, separately, by image shape
# rather than a single narrow image-count window (which previously excluded train/ entirely):
#   1. EXTRA_IMG_DIR/EXTRA_LBL_DIR   - full-size panoramics with no tooth boxes
#      (org_train_dataset: 497 images @ 1435x2943-ish). These need our trained detector to
#      pseudo-label tooth boxes before they're usable for Stage 2 (see the cell after the
#      detector is trained).
#   2. PRECROP_IMG_DIR/PRECROP_LBL_DIR - already-cropped, roughly tooth-region-sized tiles
#      with no full-panoramic context (train/: 2389 images @ 384x384, confirmed via a
#      separate exploration pass: dense content edge-to-edge, no letterboxing, low
#      adjacent-ID/near-duplicate rate, caries masks well-localized at tooth/interproximal
#      boundaries, and 384->256 resize loses no caries signal in spot checks). Running the
#      tooth detector on these would be the wrong operation - there's no panoramic context
#      for it to find a tooth box IN. Instead each record gets one synthetic full-tile box
#      [0,0,W,H], which crop_tooth()'s own padding/clipping already handles safely (the pad
#      expansion clips back to the tile edges, so this is equivalent to the CLAHE+resize
#      path already used everywhere else - no new crop code needed). This is still applying
#      CLAHE (locally-adaptive contrast) to a small pre-cropped tile rather than a full
#      panoramic, which is a mild distribution shift from how ROOT's crops are made; worth
#      knowing about, not treated as disqualifying given the validated resize/signal checks.
PANORAMIC_MIN_DIM = 900   # ROOT's panoramics are ~1435x2943; org_train_dataset's are similar scale
PRECROP_MAX_DIM = 512     # train/'s tiles are 384x384; generous ceiling to still catch similar pools

def median_min_dim(img_dir, sample=8):
    """Median of each sampled image's smaller side, to classify a pool as panoramic vs. pre-cropped."""
    files = sorted(p for p in Path(img_dir).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)[:sample]
    dims = []
    for p in files:
        g = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if g is not None:
            dims.append(min(g.shape[:2]))
    return float(np.median(dims)) if dims else 0.0

# Exclude anything literally named/under "org_test_dataset": the earlier diagnostic pass
# confirmed it mirrors the SAME 100 images as ROOT's held-out test images, just without tooth
# boxes - pseudo-labeling or training on it would leak test-set images into Stage 2 training
# under a different name. (Previously this was excluded only by accident, via the narrow
# EXTRA_MIN/EXTRA_MAX=300/700 image-count window; that protection is now explicit instead of
# incidental, since the widened size-based discovery below would otherwise catch it too.)
def is_test_mirror(folder):
    return "org_test_dataset" in str(folder).lower()

all_pool = classify(FULL_DC1000_DIR) if FULL_DC1000_DIR.exists() else []
all_pool = [c for c in all_pool if c[0] != ROOT and c[2] is None and c[3] > 0 and not is_test_mirror(c[0])]

panoramic_pool, precrop_pool = [], []
for c in all_pool:
    md = median_min_dim(c[0] / "images")
    if md >= PANORAMIC_MIN_DIM:
        panoramic_pool.append(c)
    elif 0 < md <= PRECROP_MAX_DIM:
        precrop_pool.append(c)
    # else: ambiguous size, skip rather than guess

EXTRA_INFO = max(panoramic_pool, key=lambda c: c[3]) if panoramic_pool else None
if EXTRA_INFO:
    EXTRA_IMG_DIR, EXTRA_LBL_DIR = EXTRA_INFO[0] / "images", EXTRA_INFO[0] / EXTRA_INFO[1]
    print(f"Found extra unboxed FULL-PANORAMIC caries images for Stage-2 pseudo-label augmentation: "
          f"{EXTRA_INFO[0]}  ({EXTRA_INFO[3]} images, labels='{EXTRA_INFO[1]}')")
else:
    EXTRA_IMG_DIR = EXTRA_LBL_DIR = None
    print("No extra unboxed full-panoramic caries dataset found - "
          "Stage 2's pseudo-label pool will be empty.")

PRECROP_INFO = max(precrop_pool, key=lambda c: c[3]) if precrop_pool else None
if PRECROP_INFO:
    PRECROP_IMG_DIR, PRECROP_LBL_DIR = PRECROP_INFO[0] / "images", PRECROP_INFO[0] / PRECROP_INFO[1]
    print(f"Found extra PRE-CROPPED caries tiles for direct Stage-2 augmentation: "
          f"{PRECROP_INFO[0]}  ({PRECROP_INFO[3]} images, labels='{PRECROP_INFO[1]}')")
else:
    PRECROP_IMG_DIR = PRECROP_LBL_DIR = None
    print("No extra pre-cropped caries tile pool found - skipping that augmentation source.")
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
ANN_EXT = {".json", ".txt", ".csv", ".xml", ""}  # "" supports extensionless annotation files
NAMEKEYS = {
    "file_name", "filename", "file", "image", "image_name", "img",
    "image_path", "imagepath", "path", "name", "image_id"
}

def norm(name):
    s = Path(str(name)).stem.lower().strip()
    s = re.sub(
        r"[_\-\s]*(teeth|tooth|bboxes?|boxes|labels?|masks?|ann|annotation)$",
        "", s
    )
    return str(int(s)) if s.isdigit() else s

def show_tree(root, max_depth=2):
    root = Path(root)
    print(f"== {root} ==")
    for dp, dn, fn in os.walk(root):
        dn.sort()
        depth = len(Path(dp).relative_to(root).parts)
        if depth > max_depth:
            dn[:] = []
            continue
        ext = {}
        for f in fn:
            e = Path(f).suffix.lower()
            ext[e] = ext.get(e, 0) + 1
        print("  " * depth + Path(dp).name + "/", ext, sorted(fn)[:3])

show_tree(ROOT, max_depth=2)

def n_imgs(d):
    return sum(
        1 for p in Path(d).rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXT
    )

print("\nDataset directories:")
print(f"  images: {n_imgs(IMG_DIR)} image files")
print(f"  labels: {n_imgs(LBL_DIR)} mask files")

assert n_imgs(IMG_DIR) > 0, f"No images found in {IMG_DIR}"
assert n_imgs(LBL_DIR) > 0, f"No masks found in {LBL_DIR}"

def index_dir(d):
    return {
        norm(p): p
        for p in sorted(Path(d).rglob("*"))
        if p.is_file() and p.suffix.lower() in IMG_EXT
    }

imgs = index_dir(IMG_DIR)
msks = index_dir(LBL_DIR)
keys = sorted(set(imgs) & set(msks))

if not keys and len(imgs) == len(msks) and len(imgs) > 0:
    keys = sorted(imgs)
    msks = dict(zip(sorted(imgs), [msks[k] for k in sorted(msks)]))
    print("!! Image/mask stems differ; paired by sorted order. Inspect the sample pair below.")

print(f"images={len(imgs)} masks={len(msks)} paired={len(keys)}")
assert keys, (
    f"No image<->mask pairs. Images: {list(imgs)[:5]} | "
    f"Masks: {list(msks)[:5]}"
)

def read_mask(p):
    m = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if m is None:
        return None, None
    if m.ndim == 3:
        m = m.max(axis=2)
    raw_max = float(m.max()) if m.size else 0.0
    thr = 127 if (raw_max > 1 and p.suffix.lower() in (".jpg", ".jpeg")) else 0
    return (m > thr).astype(np.uint8), np.unique(m)[:8]

recs, n_resized = [], 0
for n, k in enumerate(keys):
    g = cv2.imread(str(imgs[k]), cv2.IMREAD_GRAYSCALE)
    m, u = read_mask(msks[k])
    if g is None or m is None:
        print("Skipping unreadable pair:", imgs[k], msks[k])
        continue

    if n == 0:
        print("first image:", g.shape, g.dtype, "| first mask raw values:", u)
        print("first pair:", imgs[k], "↔", msks[k])

    if m.shape != g.shape:
        m = cv2.resize(
            m, (g.shape[1], g.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )
        n_resized += 1

    recs.append(dict(key=k, name=imgs[k].name, img=g, mask=m))

    if (n + 1) % 100 == 0:
        print(f"  loaded {n + 1}/{len(keys)}", flush=True)

print(
    f"loaded {len(recs)} image+mask pairs "
    f"(mask resized for {n_resized}); "
    f"images with any caries: {sum(int(r['mask'].any()) for r in recs)}"
)
assert recs, "No usable image+mask records were loaded."

# ------------------------------ Tooth-box annotations ------------------------------
def find_ann_files(base):
    base = Path(base)
    candidates = []

    # The dataset documents tooth bounding boxes below annotations.
    for p in ANN_DIR.rglob("*"):
        if not p.is_file():
            continue
        # Include extensionless files such as files named bboxes_teeth.
        if p.suffix.lower() in ANN_EXT:
            candidates.append(p)

    # Also tolerate an annotation file placed directly at dataset root.
    for p in base.iterdir():
        if p.is_file() and p.suffix.lower() in ANN_EXT:
            candidates.append(p)

    candidates = sorted(set(candidates))
    rel = lambda p: str(p.relative_to(base)).lower()

    # Prefer an explicitly tooth-labelled source.
    teeth = [
        p for p in candidates
        if re.search(r"teeth|tooth", rel(p)) and "caries" not in rel(p)
    ]
    if teeth:
        return teeth

    # Generic annotation files can still contain a bboxes_teeth branch.
    generic = [
        p for p in candidates
        if "annotation" in rel(p) or "bboxes" in rel(p)
    ]
    return sorted(generic)

ann_files = find_ann_files(ROOT)
print(f"tooth-annotation files found: {len(ann_files)}")
for p in ann_files[:10]:
    print("  ", p)

assert ann_files, (
    f"No tooth-box annotation files found under {ANN_DIR}. "
    "Expected something like bboxes_teeth.* or an annotation JSON containing bboxes_teeth."
)


box-annotated (images+labels+annotations) candidates under /kaggle/input: [('/kaggle/input/datasets/testpeaky/dental', 100)]

Only 100 box-annotated images available as a Kaggle Input - fetching the full DC1000 release from Google Drive to check for more (needs Internet ON)...


Downloading...
From (original): https://drive.google.com/uc?id=1Xn1oGHvhGF9GbkcLEtCOV5QvWWqt1y62
From (redirected): https://drive.google.com/uc?id=1Xn1oGHvhGF9GbkcLEtCOV5QvWWqt1y62&confirm=t&uuid=dc8b18b7-a5d3-4698-a924-43c28bc264e5
To: /kaggle/working/dc1000_full/dc1000.zip
100%|██████████| 1.10G/1.10G [00:20<00:00, 53.7MB/s]


box-annotated candidates after download: []
  The full release doesn't add a bigger box-annotated set (it only ships unboxed caries-segmentation images - see EXTRA_* below). Keeping the existing box-annotated Input for the detector and its share of Stage 2.

Using box-annotated dataset root: /kaggle/input/datasets/testpeaky/dental  (100 images, labels='labels', annotations=/kaggle/input/datasets/testpeaky/dental/annotations)
Found extra unboxed FULL-PANORAMIC caries images for Stage-2 pseudo-label augmentation: /kaggle/working/dc1000_full/DC1000_dataset/org_train_dataset  (497 images, labels='labels_clean')
Found extra PRE-CROPPED caries tiles for direct Stage-2 augmentation: /kaggle/working/dc1000_full/DC1000_dataset/train  (2389 images, labels='labels')
== /kaggle/input/datasets/testpeaky/dental ==
dental/ {} []
  annotations/ {} []
    bboxes_caries/ {'.txt': 100} ['1008.txt', '1009.txt', '1016.txt']
    bboxes_teeth/ {'.txt': 100} ['1008.txt', '1009.txt', '1016.txt']
  images/ {'.p

In [5]:
# ---- tooth-box annotations: parse whatever format they are in ------------------------------------------------
def _num(x): return isinstance(x, (int, float)) and not isinstance(x, bool)
def _f(v):
    try: return float(v)
    except Exception: return v

def extract_boxes(obj, out):
    """Pull every box out of an arbitrary JSON-ish object. out: list of ([4 numbers], kind), kind in xyxy|xywh|raw."""
    if isinstance(obj, dict):
        lk = {str(k).lower(): k for k in obj}
        g = lambda *ks: [obj[lk[k]] for k in ks]
        if all(k in lk for k in ("xmin", "ymin", "xmax", "ymax")):
            out.append((g("xmin", "ymin", "xmax", "ymax"), "xyxy")); return
        if all(k in lk for k in ("x1", "y1", "x2", "y2")):
            out.append((g("x1", "y1", "x2", "y2"), "xyxy")); return
        for wh in (("width", "height"), ("w", "h")):
            if all(k in lk for k in ("x", "y") + wh):
                out.append((g("x", "y", *wh), "xywh")); return
        if "points" in lk:
            try:
                pts = np.asarray(obj[lk["points"]], float).reshape(-1, 2)
                out.append(([pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max()], "xyxy")); return
            except Exception:
                pass
        for k in ("bbox", "box", "bndbox", "bounding_box", "boundingbox"):
            if k in lk:
                v = obj[lk[k]]
                if isinstance(v, (list, tuple)) and len(v) == 4 and all(_num(a) for a in v):
                    out.append((list(v), "raw")); return
                extract_boxes(v, out); return
        for v in obj.values(): extract_boxes(v, out)
    elif isinstance(obj, (list, tuple)):
        if len(obj) == 4 and all(_num(a) for a in obj): out.append((list(obj), "raw"))
        else:
            for v in obj: extract_boxes(v, out)

def keep_teeth(obj):
    """If an annotation blob holds both tooth and caries boxes (e.g. keys bboxes_teeth / bboxes_caries), keep only teeth."""
    if isinstance(obj, dict):
        tk = [k for k in obj if isinstance(k, str) and re.search(r"teeth|tooth", k, re.I) and "caries" not in k.lower()]
        if tk: return obj[tk[0]] if len(tk) == 1 else {k: obj[k] for k in tk}
        return {k: keep_teeth(v) for k, v in obj.items()}
    if isinstance(obj, list): return [keep_teeth(v) for v in obj]
    return obj

def load_ann_file(p):
    p = Path(p)
    s = p.suffix.lower()

    if s == ".json":
        with open(p, encoding="utf-8") as f:
            return keep_teeth(json.load(f))

    if s == ".csv":
        with open(p, newline="", encoding="utf-8") as f:
            return [
                {(k or "").strip().lower(): _f(v) for k, v in row.items()}
                for row in csv.DictReader(f)
            ]

    if s == ".xml":
        import xml.etree.ElementTree as ET
        out = []
        for bb in ET.parse(p).getroot().iter("bndbox"):
            v = {c.tag.lower(): float(c.text) for c in bb}
            out.append(([v["xmin"], v["ymin"], v["xmax"], v["ymax"]], "xyxy"))
        return out

    # .txt or extensionless:
    # First try JSON, because a Kaggle copy may preserve an annotation filename
    # without a conventional suffix.
    raw = p.read_text(encoding="utf-8", errors="ignore").strip()
    if raw:
        try:
            return keep_teeth(json.loads(raw))
        except Exception:
            pass

    out = []  # plain text: x1 y1 x2 y2 OR cls x1 y1 x2 y2
    for line in raw.splitlines():
        nums = [
            float(t)
            for t in re.findall(
                r"-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", line
            )
        ]
        if len(nums) == 4:
            out.append((nums, "raw"))
        elif len(nums) == 5:
            out.append((nums[1:], "raw"))
    return out

def boxes_from(obj):
    if isinstance(obj, list) and obj and isinstance(obj[0], tuple) and len(obj[0]) == 2 and isinstance(obj[0][1], str):
        return obj
    out = []; extract_boxes(obj, out); return out

def index_annotations(obj):
    """One big annotation file -> {norm image stem: [(box, kind), ...]}  (COCO, dict-by-name, list-of-records, CSV rows)."""
    res = collections.defaultdict(list)
    if isinstance(obj, dict):
        if "images" in obj and "annotations" in obj:                       # COCO
            id2name = {im.get("id"): (im.get("file_name") or im.get("filename")) for im in obj["images"]}
            for a in obj["annotations"]:
                nm = id2name.get(a.get("image_id"))
                if nm is not None: res[norm(nm)].extend(boxes_from(a))
        else:
            for k, v in obj.items():
                b = boxes_from(v)
                if b: res[norm(k)].extend(b)
    elif isinstance(obj, list):
        for it in obj:
            if not isinstance(it, dict): continue
            nm = next((it[k] for k in it if str(k).lower() in NAMEKEYS and isinstance(it[k], (str, int, float))), None)
            if nm is None: continue
            rest = {k: v for k, v in it.items() if str(k).lower() not in NAMEKEYS}
            res[norm(nm)].extend(boxes_from(rest))
    return res

def find_ann_files(base):
    fs = [p for p in base.rglob("*") if p.is_file() and p.suffix.lower() in ANN_EXT]
    rel = lambda p: str(p.relative_to(base)).lower()
    t = [p for p in fs if re.search(r"teeth|tooth", rel(p)) and "caries" not in rel(p)]
    if t: return sorted(t)
    return sorted(p for p in fs if "annot" in rel(p) and "caries" not in rel(p))

ann_files = find_ann_files(ROOT)
if not ann_files and ROOT.parent != IN: ann_files = find_ann_files(ROOT.parent)
print(f"tooth-annotation files found: {len(ann_files)}")
for p in ann_files[:5]: print("  ", p)
assert ann_files, "No tooth-box annotation files found - see the tree printed above and paste it."

RAW = collections.defaultdict(list)
per_image = np.mean([norm(p) in imgs for p in ann_files]) > 0.5
print("mode:", "one annotation file per image" if per_image else "annotation file(s) covering many images")
for p in ann_files:
    obj = load_ann_file(p)
    if per_image: RAW[norm(p)].extend(boxes_from(obj))
    else:
        for k, v in index_annotations(obj).items(): RAW[k].extend(v)

n_before = len(recs)
recs = [r for r in recs if RAW.get(r["key"])]
for r in recs: r["raw"] = RAW[r["key"]]
print(f"images with tooth boxes: {len(recs)}/{n_before}")
if not recs:
    print("sample annotation content:", repr(load_ann_file(ann_files[0]))[:800])
    raise RuntimeError("Could not map annotations to images. Paste the 'sample annotation content' + tree above.")

# ---- decide the box convention (xyxy / xywh / cxcywh) by checking boxes against caries masks -------------------
def to_xyxy(a, mode, W, H, normed):
    a = np.asarray(a, np.float32).reshape(-1, 4).copy()
    if normed: a *= np.array([W, H, W, H], np.float32)
    if mode == "xywh":
        a[:, 2] += a[:, 0]; a[:, 3] += a[:, 1]
    elif mode == "cxcywh":
        cx, cy, w, h = a.T.copy(); a = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)
    return a

kinds = collections.Counter(k for r in recs for _, k in r["raw"])
normed = max(max(b) for r in recs for b, _ in r["raw"]) <= 1.5
explicit = [k for k in kinds if k in ("xyxy", "xywh")]
sample = [r for r in recs if r["mask"].any()][:60] or recs[:60]

def score_mode(mode):
    cov, inval, oob, area = [], [], [], []
    for r in sample:
        H, W = r["img"].shape
        b = to_xyxy([x for x, _ in r["raw"]], mode, W, H, normed)
        inval.append(np.mean((b[:, 2] - b[:, 0] < 3) | (b[:, 3] - b[:, 1] < 3)))
        oob.append(np.mean((b[:, 2] > W * 1.02) | (b[:, 3] > H * 1.02) | (b[:, 0] < -W * .02) | (b[:, 1] < -H * .02)))
        cm = np.zeros((H, W), np.uint8)
        for x1, y1, x2, y2 in np.clip(b, 0, [W, H, W, H]).astype(int): cm[y1:y2, x1:x2] = 1
        area.append(cm.mean())
        if r["mask"].any(): cov.append(float((r["mask"] & cm).sum() / r["mask"].sum()))
    c = float(np.mean(cov)) if cov else 0.0
    return c - np.mean(inval) - np.mean(oob) - 0.5 * np.mean(area), c

if len(explicit) == 1 and "raw" not in kinds:
    MODE = explicit[0]; print("box convention from keys:", MODE)
else:
    sc = {m: score_mode(m) for m in ("xyxy", "xywh", "cxcywh")}
    print("box-convention scores (score, mask-coverage):", {m: (round(v[0], 3), round(v[1], 3)) for m, v in sc.items()})
    MODE = max(sc, key=lambda m: sc[m][0]); print("-> using:", MODE)
print("normalized coordinates:", normed)

for r in recs:
    H, W = r["img"].shape
    b = to_xyxy([x for x, _ in r["raw"]], MODE, W, H, normed)
    b[:, [0, 2]] = b[:, [0, 2]].clip(0, W - 1); b[:, [1, 3]] = b[:, [1, 3]].clip(0, H - 1)
    b = b[((b[:, 2] - b[:, 0]) >= 6) & ((b[:, 3] - b[:, 1]) >= 6)]
    r["boxes"] = b.astype(np.float32); del r["raw"]
recs = [r for r in recs if len(r["boxes"]) > 0]

# sanity: how much of the caries mask do the tooth boxes cover (upper bound for the 2-stage pipeline)?
tot = cov_px = 0
for r in recs:
    cm = np.zeros_like(r["mask"])
    for x1, y1, x2, y2 in r["boxes"].astype(int): cm[y1:y2, x1:x2] = 1
    tot += int(r["mask"].sum()); cov_px += int((r["mask"] & cm).sum())
COVER = cov_px / max(tot, 1)
nb = np.array([len(r["boxes"]) for r in recs]); wh = np.concatenate([r["boxes"][:, 2:] - r["boxes"][:, :2] for r in recs])
print(f"{len(recs)} images | teeth/img mean {nb.mean():.1f} (min {nb.min()}, max {nb.max()}) | median tooth box {np.median(wh[:,0]):.0f}x{np.median(wh[:,1]):.0f}px | caries px inside tooth boxes: {COVER:.1%}")
if COVER < 0.5:
    print("first record boxes:", recs[0]["boxes"][:3], "| img shape:", recs[0]["img"].shape)
    raise RuntimeError("Tooth boxes do not cover the caries masks - box convention/parsing is wrong. Paste this cell's output.")

# ---- split ------------------------------------------------------------------------------------------------------
perm = np.random.RandomState(CFG["seed"]).permutation(len(recs))
n_te = max(1, round(CFG["test_frac"] * len(recs))); n_va = max(1, round(CFG["val_frac"] * len(recs)))
test = [recs[i] for i in perm[:n_te]]; val = [recs[i] for i in perm[n_te:n_te + n_va]]; train = [recs[i] for i in perm[n_te + n_va:]]
print(f"split: train {len(train)} | val {len(val)} | test {len(test)}  (elapsed {elapsed_h():.2f} h)")


tooth-annotation files found: 100
   /kaggle/input/datasets/testpeaky/dental/annotations/bboxes_teeth/1008.txt
   /kaggle/input/datasets/testpeaky/dental/annotations/bboxes_teeth/1009.txt
   /kaggle/input/datasets/testpeaky/dental/annotations/bboxes_teeth/1016.txt
   /kaggle/input/datasets/testpeaky/dental/annotations/bboxes_teeth/1018.txt
   /kaggle/input/datasets/testpeaky/dental/annotations/bboxes_teeth/1026.txt
mode: one annotation file per image
images with tooth boxes: 100/100
box-convention scores (score, mask-coverage): {'xyxy': (np.float64(0.914), 0.997), 'xywh': (np.float64(-0.02), 1.0), 'cxcywh': (np.float64(0.465), 1.0)}
-> using: xyxy
normalized coordinates: False
100 images | teeth/img mean 30.2 (min 26, max 33) | median tooth box 96x245px | caries px inside tooth boxes: 98.9%
split: train 80 | val 10 | test 10  (elapsed 0.02 h)


## 2. Stage 1 - Faster R-CNN tooth detector (2-GPU DDP)

Stage 1 uses PyTorch DistributedDataParallel (DDP): one persistent Faster R-CNN process/model per Tesla T4. This avoids the repeated model replication and Python-thread overhead of the previous DataParallel implementation.

Detector images are loaded lazily from disk inside each process. `num_workers=0` and `pin_memory=False` are intentional for Kaggle host-RAM stability.

If `/kaggle/working/det_best.pt` already exists, it is reused automatically. Otherwise the notebook launches `torchrun --nproc_per_node=2`.

In [6]:
from torchvision.ops import box_iou
import subprocess, os, gc, pickle

DET_MIN, DET_MAX = 800, 1700
DET_SCRIPT = OUT / "train_detector_ddp.py"
DET_META = OUT / "det_ddp_meta.pkl"

det_meta={
    "train":[{"image":str(IMG_DIR/r["name"]),"boxes":r["boxes"].astype(np.float32)} for r in train],
    "val":[{"image":str(IMG_DIR/r["name"]),"boxes":r["boxes"].astype(np.float32)} for r in val],
    "cfg":{"epochs":int(CFG["det_epochs"]),"batch_size":int(CFG["det_bs"]),"lr":float(CFG["det_lr"]),"patience":int(CFG["det_patience"]),"det_budget_h":float(CFG["det_budget_h"]),"seed":int(CFG["seed"]),"min_size":DET_MIN,"max_size":DET_MAX},
}
with open(DET_META,"wb") as f: pickle.dump(det_meta,f,protocol=4)
print("DDP metadata:",DET_META,DET_META.stat().st_size,"bytes")


DDP metadata: /kaggle/working/det_ddp_meta.pkl 52800 bytes


In [7]:
%%writefile /kaggle/working/train_detector_ddp.py
import os, gc, math, time, random, pickle
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models import MobileNet_V3_Large_Weights
from torchvision.ops import box_iou

def seed_all(seed, rank):
    s=seed+rank
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def make_scaler():
    try: return torch.amp.GradScaler('cuda',enabled=True)
    except (AttributeError,TypeError): return torch.cuda.amp.GradScaler(enabled=True)

class DetDS(Dataset):
    def __init__(self, records): self.records=records
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        r=self.records[i]; g=cv2.imread(r['image'],cv2.IMREAD_GRAYSCALE)
        if g is None: raise IOError(r['image'])
        b=np.asarray(r['boxes'],np.float32).copy()
        if random.random()<0.5:
            g=np.ascontiguousarray(g[:,::-1]); b[:,[0,2]]=g.shape[1]-b[:,[2,0]]
        g=cv2.convertScaleAbs(np.ascontiguousarray(g),alpha=random.uniform(.75,1.25),beta=random.uniform(-25,25))
        x=torch.from_numpy(np.ascontiguousarray(g)).float().div_(255).unsqueeze(0).repeat(3,1,1)
        t={'boxes':torch.from_numpy(b).float(),'labels':torch.ones(len(b),dtype=torch.int64)}
        return x,t

def collate(batch): return tuple(zip(*batch))

def make_detector(min_size,max_size):
    m=fasterrcnn_mobilenet_v3_large_fpn(weights=None,weights_backbone=MobileNet_V3_Large_Weights.IMAGENET1K_V1,num_classes=2,min_size=min_size,max_size=max_size,box_detections_per_img=64)
    m.transform.min_size=(640,704,768,800)
    return m

@torch.no_grad()
def evaluate(model,records,device,thr=.5):
    model.eval(); tp=fp=fn=0; ious=[]
    for r in records:
        img=cv2.imread(r['image'],cv2.IMREAD_GRAYSCALE)
        x=torch.from_numpy(np.ascontiguousarray(img)).float().div_(255).unsqueeze(0).repeat(3,1,1).to(device)
        o=model([x])[0]; pb=o['boxes'].float().cpu(); ps=o['scores'].float().cpu(); k=ps>=thr; pb=pb[k]; ps=ps[k]
        gb=torch.from_numpy(np.asarray(r['boxes'],np.float32))
        if len(pb)==0: fn+=len(gb); continue
        iou=box_iou(pb,gb); used=torch.zeros(len(gb),dtype=torch.bool)
        for pi in ps.argsort(descending=True).tolist():
            row=iou[pi].clone(); row[used]=-1; j=int(row.argmax())
            if row[j]>=.5: tp+=1; used[j]=True; ious.append(float(row[j]))
            else: fp+=1
        fn+=int((~used).sum())
    p=tp/max(tp+fp,1); r=tp/max(tp+fn,1)
    return {'precision':p,'recall':r,'f1':2*p*r/max(p+r,1e-9),'mean_iou':float(np.mean(ious)) if ious else 0.0,'tp':tp,'fp':fp,'fn':fn}

def main():
    local_rank=int(os.environ['LOCAL_RANK']); rank=int(os.environ['RANK']); world=int(os.environ['WORLD_SIZE'])
    torch.cuda.set_device(local_rank); device=torch.device(f'cuda:{local_rank}')
    dist.init_process_group(backend='nccl',init_method='env://')
    with open(os.environ['DET_META_PATH'],'rb') as f: meta=pickle.load(f)
    cfg=meta['cfg']; seed_all(cfg['seed'],rank)
    ds=DetDS(meta['train'])
    sampler=DistributedSampler(ds,num_replicas=world,rank=rank,shuffle=True,drop_last=True,seed=cfg['seed'])
    loader=DataLoader(ds,batch_size=cfg['batch_size'],sampler=sampler,num_workers=0,pin_memory=False,collate_fn=collate,drop_last=True)
    model=make_detector(cfg['min_size'],cfg['max_size']).to(device)
    ddp=DDP(model,device_ids=[local_rank],output_device=local_rank,broadcast_buffers=False,find_unused_parameters=False)
    opt=torch.optim.SGD(ddp.parameters(),lr=cfg['lr'],momentum=.9,weight_decay=1e-4,nesterov=True)
    total=max(1,cfg['epochs']*len(loader)); warm=max(1,min(300,total//10))
    def lf(step):
        if step<warm: return .001+.999*step/warm
        return .02+.98*.5*(1+math.cos(math.pi*(step-warm)/max(1,total-warm)))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,lf); scaler=make_scaler(); best=-1; bad=0; stage=time.time()
    for ep in range(cfg['epochs']):
        sampler.set_epoch(ep); ddp.train(); t0=time.time(); tsum=0.; n=0
        for imgs,tgts in loader:
            imgs=[x.to(device) for x in imgs]
            tgts=[{k:(v.to(device) if torch.is_tensor(v) else v) for k,v in t.items()} for t in tgts]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16): loss=sum(ddp(imgs,tgts).values())
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(ddp.parameters(),10.0); scaler.step(opt); scaler.update(); sched.step()
            tsum+=float(loss.detach()); n+=1
        dist.barrier()
        if rank==0:
            val=evaluate(ddp.module,meta['val'],device,.5)
            if val['f1']>best:
                best=val['f1']; bad=0; torch.save({'state_dict':{k:v.detach().cpu() for k,v in ddp.module.state_dict().items()},'epoch':ep+1,'val':val},'/kaggle/working/det_best.pt')
            else: bad+=1
            peak=torch.cuda.max_memory_allocated(device)/(1024**3)
            print(f"[det-DDP] ep {ep+1:3d}/{cfg['epochs']} loss {tsum/max(n,1):.4f} | val P {val['precision']:.3f} R {val['recall']:.3f} F1 {val['f1']:.3f} IoU {val['mean_iou']:.3f} | best {best:.3f} | peak VRAM {peak:.2f}G | {time.time()-t0:.0f}s",flush=True)
            stop=(bad>=cfg['patience'] or (time.time()-stage)/3600>cfg['det_budget_h'])
        else: stop=False
        flag=torch.tensor([int(stop)],device=device,dtype=torch.int32); dist.broadcast(flag,src=0); gc.collect(); torch.cuda.empty_cache()
        if bool(flag.item()): break
    dist.barrier()
    if rank==0: print('DDP detector complete')
    dist.destroy_process_group()

if __name__=='__main__': main()


Writing /kaggle/working/train_detector_ddp.py


In [8]:
from torchvision.ops import box_iou
# Detector evaluation metrics used after the DDP training stage.
def det_metrics(preds, recs, thr=0.5, iou_thr=0.5):
    tp = fp = fn = 0
    ious = []

    for (pb, ps), r in zip(preds, recs):
        k = ps >= thr
        pb, ps = pb[k], ps[k]
        gb = torch.from_numpy(r["boxes"]).float()

        if len(pb) == 0:
            fn += len(gb)
            continue

        if len(gb) == 0:
            fp += len(pb)
            continue

        iou = box_iou(pb, gb)
        used = torch.zeros(len(gb), dtype=torch.bool)

        for pi in ps.argsort(descending=True).tolist():
            row = iou[pi].clone()
            row[used] = -1
            j = int(row.argmax())

            if row[j] >= iou_thr:
                tp += 1
                used[j] = True
                ious.append(float(row[j]))
            else:
                fp += 1

        fn += int((~used).sum())

    p = tp / max(tp + fp, 1)
    rc = tp / max(tp + fn, 1)

    return dict(
        precision=p,
        recall=rc,
        f1=2 * p * rc / max(p + rc, 1e-9),
        mean_iou=float(np.mean(ious)) if ious else 0.0,
        tp=tp,
        fp=fp,
        fn=fn,
    )


In [9]:
import subprocess, os, gc, torch

if CFG["reuse_ckpt"] and (OUT/"det_best.pt").exists():
    print("Reusing existing det_best.pt; skipping detector training.")
else:
    env=os.environ.copy(); env["DET_META_PATH"]=str(DET_META)
    result=subprocess.run(["torchrun","--standalone","--nproc_per_node=2",str(DET_SCRIPT)],env=env,check=False)
    if result.returncode!=0: raise RuntimeError(f"DDP detector exited with code {result.returncode}")

if not (OUT/"det_best.pt").exists(): raise FileNotFoundError("det_best.pt was not produced")

det_core,_=build_detector(pretrained=False,min_size=DET_MIN,max_size=DET_MAX)
det_core.to(dev)
ckpt=torch.load(OUT/"det_best.pt",map_location="cpu")
state=ckpt["state_dict"] if isinstance(ckpt,dict) and "state_dict" in ckpt else ckpt
det_core.load_state_dict(state); det_core.transform.min_size=(DET_MIN,); det_core.eval()
print("Loaded detector checkpoint. Device:",next(det_core.parameters()).device)

@torch.no_grad()
def det_predict_ddp(model,recs,bs=2):
    model.eval(); out=[]
    for i in range(0,len(recs),bs):
        xs=[torch.from_numpy(r["img"]).to(dev).float().div(255).unsqueeze(0).repeat(3,1,1) for r in recs[i:i+bs]]
        for o in model(xs): out.append((o["boxes"].float().cpu(),o["scores"].float().cpu()))
    return out

pv=det_predict_ddp(det_core,val)
sweep={t:det_metrics(pv,val,t) for t in (0.3,0.4,0.5,0.6,0.7,0.8)}
DET_THR=max(sweep,key=lambda t:sweep[t]["f1"]); DET_TEST=det_metrics(det_predict_ddp(det_core,test),test,DET_THR)
print("DET_THR",DET_THR,"TEST",DET_TEST)
gc.collect(); torch.cuda.empty_cache()


W0922 02:05:50.283000 198 torch/distributed/run.py:852] 
W0922 02:05:50.283000 198 torch/distributed/run.py:852] *****************************************
W0922 02:05:50.283000 198 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0922 02:05:50.283000 198 torch/distributed/run.py:852] *****************************************
[W922 02:05:50.494065355 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W922 02:05:53.940606470 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W922 02:05:53.941592568 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
  0%|          | 0.00/21.1M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 201MB/s]
100%|██████████| 21.1M/21.1M [00:00<00:00, 153MB/s]
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


[det-DDP] ep   1/60 loss 1.7427 | val P 0.000 R 0.000 F1 0.000 IoU 0.000 | best 0.000 | peak VRAM 0.96G | 48s
[det-DDP] ep   2/60 loss 1.7589 | val P 0.281 R 0.477 F1 0.354 IoU 0.631 | best 0.354 | peak VRAM 0.96G | 15s
[det-DDP] ep   3/60 loss 1.4557 | val P 0.451 R 0.662 F1 0.537 IoU 0.640 | best 0.537 | peak VRAM 0.96G | 7s
[det-DDP] ep   4/60 loss 1.3136 | val P 0.371 R 0.772 F1 0.501 IoU 0.620 | best 0.537 | peak VRAM 0.96G | 7s
[det-DDP] ep   5/60 loss 1.2131 | val P 0.397 R 0.805 F1 0.532 IoU 0.644 | best 0.537 | peak VRAM 0.96G | 7s
[det-DDP] ep   6/60 loss 1.1280 | val P 0.443 R 0.818 F1 0.575 IoU 0.673 | best 0.575 | peak VRAM 0.96G | 7s
[det-DDP] ep   7/60 loss 1.0766 | val P 0.444 R 0.861 F1 0.586 IoU 0.670 | best 0.586 | peak VRAM 0.96G | 7s
[det-DDP] ep   8/60 loss 1.0484 | val P 0.498 R 0.917 F1 0.646 IoU 0.688 | best 0.646 | peak VRAM 0.96G | 7s
[det-DDP] ep   9/60 loss 1.0089 | val P 0.578 R 0.887 F1 0.700 IoU 0.694 | best 0.700 | peak VRAM 0.96G | 7s
[det-DDP] ep  10/

## 3. Stage 2 - U-Net caries segmentation (2-GPU DDP + RAM-safe crops)

The U-Net stage uses true `DistributedDataParallel` with one process per T4.
Training crops are stored as disk-backed NumPy memory maps, not as ordinary arrays
inside each process.

This replaces the previous `nn.DataParallel` implementation because that run showed
host-RAM growth from ~6 GB to ~20 GB after only two epochs while taking 580 s and 454 s
per epoch.

The new configuration uses 32 crops per GPU (64 global batch), `num_workers=0`,
and `pin_memory=False`. Each epoch samples 620 positive + 620 negative crops.


The DDP wrapper enables `find_unused_parameters=True` because the current EfficientNet-B0 U-Net configuration reports a small set of parameters that do not receive gradients on every iteration.


### False-positive reduction changes (Stage 2)
**Needs Stage-2 retraining** (Stage 1 is untouched): Dice-balanced loss (was Tversky 0.3/0.7) plus an empty-crop activation penalty; periodic hard-negative mining (the crops the model still fires on replace half of the random negative quota each epoch); checkpoint chosen by *Dice minus empty-crop FP rate* on a larger validation set (box-annotated val + held-out pseudo-labeled panoramics + a held-out block of pre-cropped tiles).

**Inference-only** (no retraining; stored in the pickle): each crop votes only inside its own tooth box and overlapping votes are averaged (was max over padded crops); hysteresis threshold + minimum blob area, tuned on the larger validation set with a lesion-level objective.

Section 4 reports the new settings next to the legacy inference behaviour on the same weights, plus lesion-level and tooth-level false-positive metrics.

In [ ]:
# ------------------------------ Stage 2 extra data: pseudo-labeled + pre-cropped ------------------------------
# Two extra data sources get folded into Stage 2's training pool, using two different
# mechanisms, because they're structurally different (see the discovery cell above):
#
#  1. EXTRA_IMG_DIR (full panoramics, no tooth boxes): our own trained detector (F1/IoU
#     already validated above) pseudo-labels tooth boxes onto them.
#  2. PRECROP_IMG_DIR (already tooth-region-sized tiles, e.g. train/): no detector pass -
#     each tile gets ONE synthetic full-tile box [0,0,W,H]. crop_tooth()'s own padding/
#     clipping (CFG["pad"]) safely no-ops past the tile edges, so this reuses the exact same
#     crop_tooth/crop_mask path as every other record - just skipping the "find the box"
#     step because these tiles already ARE the box. Validated in a separate exploration pass:
#     dense content edge-to-edge (no letterboxing), <1% near-duplicate rate, masks
#     well-localized at plausible caries sites, and 384->256 resize loses no caries signal
#     in spot checks.
#
# Both stay Stage-2-training-only - val/test remain exactly the original box-annotated split,
# so reported metrics stay honest (no pseudo-labels or pre-cropped tiles in eval).
SEG_TRAIN_RECS = list(train)
n_after_boxed = len(SEG_TRAIN_RECS)
# Stage-2 validation set: the 10 box-annotated val images PLUS held-out images from the extra pools
# (they have caries masks too). 302 crops from 10 images was far too small/noisy to measure false positives
# or to pick thresholds. Held-out records are removed from Stage-2 TRAINING; test stays box-annotated only.
SEG_VAL_RECS = list(val)
_rs_val = np.random.RandomState(CFG["seed"] + 1)

if EXTRA_IMG_DIR is not None:
    eimgs, emsks = index_dir(EXTRA_IMG_DIR), index_dir(EXTRA_LBL_DIR)
    ekeys = sorted(set(eimgs) & set(emsks))
    print(f"extra unboxed panoramic pool: {len(eimgs)} images, {len(emsks)} masks, {len(ekeys)} paired")

    extra_raw = []
    for k in ekeys:
        g = cv2.imread(str(eimgs[k]), cv2.IMREAD_GRAYSCALE)
        m, _ = read_mask(emsks[k])
        if g is None or m is None:
            continue
        if m.shape != g.shape:
            m = cv2.resize(m, (g.shape[1], g.shape[0]), interpolation=cv2.INTER_NEAREST)
        extra_raw.append(dict(key=k, name=eimgs[k].name, img=g, mask=m))
    print(f"extra usable image+mask pairs: {len(extra_raw)}")

    pe = det_predict_ddp(det_core, extra_raw, bs=2)
    extra_train = []
    for (pb, ps), r in zip(pe, extra_raw):
        keep = ps >= DET_THR
        b = pb[keep].numpy().astype(np.float32)
        if len(b) == 0:
            continue
        r2 = dict(r); r2["boxes"] = b
        extra_train.append(r2)

    n_extra_boxes = sum(len(r["boxes"]) for r in extra_train)
    print(f"pseudo-labeled {len(extra_train)}/{len(extra_raw)} extra images "
          f"({n_extra_boxes} pseudo tooth boxes, DET_THR={DET_THR}) - added to Stage-2 training only")
    _nv = min(CFG["seg_extra_val_imgs"], len(extra_train) // 5)
    _hold = set(_rs_val.permutation(len(extra_train))[:_nv].tolist())      # whole panoramics, so no crop-level leakage
    SEG_VAL_RECS += [r for i, r in enumerate(extra_train) if i in _hold]
    extra_train = [r for i, r in enumerate(extra_train) if i not in _hold]
    print(f"held out {_nv} pseudo-labeled panoramics for Stage-2 validation")
    SEG_TRAIN_RECS += extra_train
else:
    print("No extra unboxed panoramic dataset available - skipping pseudo-label augmentation.")

n_after_pseudo = len(SEG_TRAIN_RECS)

if PRECROP_IMG_DIR is not None:
    pimgs, pmsks = index_dir(PRECROP_IMG_DIR), index_dir(PRECROP_LBL_DIR)
    pkeys = sorted(set(pimgs) & set(pmsks))
    print(f"\nextra pre-cropped tile pool: {len(pimgs)} images, {len(pmsks)} masks, {len(pkeys)} paired")

    precrop_train = []
    n_bad = 0
    for k in pkeys:
        g = cv2.imread(str(pimgs[k]), cv2.IMREAD_GRAYSCALE)
        m, _ = read_mask(pmsks[k])
        if g is None or m is None:
            n_bad += 1
            continue
        if m.shape != g.shape:
            m = cv2.resize(m, (g.shape[1], g.shape[0]), interpolation=cv2.INTER_NEAREST)
        H, W = g.shape[:2]
        # synthetic full-tile box: crop_tooth's own pad+clip makes this equivalent to a
        # plain CLAHE+resize of the whole tile, reusing the exact same downstream code path.
        box = np.array([[0.0, 0.0, float(W), float(H)]], dtype=np.float32)
        precrop_train.append(dict(key=k, name=pimgs[k].name, img=g, mask=m, boxes=box))
    if n_bad:
        print(f"  skipped {n_bad} unreadable pre-cropped pairs")
    print(f"pre-cropped tiles added directly (no detector pass, 1 synthetic box each): "
          f"{len(precrop_train)}/{len(pkeys)} - added to Stage-2 training only")
    _nt = min(CFG["seg_extra_val_tiles"], len(precrop_train) // 5)
    _s0 = int(_rs_val.randint(0, len(precrop_train) - _nt + 1))            # one CONTIGUOUS block: neighbouring tiles may share a source
    SEG_VAL_RECS += precrop_train[_s0:_s0 + _nt]
    precrop_train = precrop_train[:_s0] + precrop_train[_s0 + _nt:]
    print(f"held out {_nt} contiguous pre-cropped tiles for Stage-2 validation")
    SEG_TRAIN_RECS += precrop_train
else:
    print("\nNo extra pre-cropped tile pool available - skipping that augmentation source.")

n_pseudo = n_after_pseudo - n_after_boxed
n_precrop = len(SEG_TRAIN_RECS) - n_after_pseudo
print(f"\nStage-2 training pool: {len(train)} box-annotated + {n_pseudo} pseudo-labeled + "
      f"{n_precrop} pre-cropped = {len(SEG_TRAIN_RECS)} images "
      f"| Stage-2 val: {len(SEG_VAL_RECS)} records ({len(val)} box-annotated + held-out extras); "
      f"test unchanged at {len(test)} (box-annotated only)")


In [ ]:

# ------------------------------ Stage 2: RAM-safe 2-GPU DDP U-Net
import os
import gc
import pickle
import subprocess
from pathlib import Path

S, PAD = CFG["crop"], CFG["pad"]

SEG_ROOT = OUT / "seg_memmap_v2"   # new folder: val set / sampling changed, so never reuse v1 crop caches
SEG_ROOT.mkdir(exist_ok=True)

SEG_XTR = SEG_ROOT / "Xtr.npy"
SEG_YTR = SEG_ROOT / "Ytr.npy"
SEG_XVA = SEG_ROOT / "Xva.npy"
SEG_YVA = SEG_ROOT / "Yva.npy"
SEG_POS = SEG_ROOT / "pos.npy"
SEG_NEG = SEG_ROOT / "neg.npy"
SEG_META = OUT / "seg_ddp_meta.pkl"
SEG_SCRIPT = OUT / "train_unet_ddp.py"


def count_crops(rs):
    n = 0
    for r in rs:
        for b in r["boxes"]:
            c, _ = crop_tooth(r["img"], b, S, PAD)
            if c is not None:
                n += 1
    return n


def write_crop_memmaps(rs, x_path, y_path):
    n = count_crops(rs)

    X = np.lib.format.open_memmap(
        x_path, mode="w+", dtype=np.uint8, shape=(n, S, S)
    )
    Y = np.lib.format.open_memmap(
        y_path, mode="w+", dtype=np.uint8, shape=(n, S, S)
    )

    j = 0
    for r in rs:
        for b in r["boxes"]:
            c, pb = crop_tooth(r["img"], b, S, PAD)
            if c is None:
                continue
            X[j] = c
            Y[j] = crop_mask(r["mask"], pb, S)
            j += 1

    X.flush()
    Y.flush()
    del X, Y
    return n


if not (SEG_XTR.exists() and SEG_YTR.exists() and SEG_XVA.exists() and SEG_YVA.exists()):
    t0 = time.time()
    ntr = write_crop_memmaps(SEG_TRAIN_RECS, SEG_XTR, SEG_YTR)
    nva = write_crop_memmaps(SEG_VAL_RECS, SEG_XVA, SEG_YVA)
    print(
        f"Created memory-mapped crops: train {ntr} | val {nva} | "
        f"{time.time() - t0:.0f}s"
    )
else:
    x0 = np.load(SEG_XTR, mmap_mode="r")
    x1 = np.load(SEG_XVA, mmap_mode="r")
    print(
        f"Reusing memory-mapped crops: train {len(x0)} | val {len(x1)}"
    )
    del x0, x1


# Determine positive and negative crops without keeping the crop images in RAM.
Ytr_mm = np.load(SEG_YTR, mmap_mode="r")
flat_sums = np.asarray(Ytr_mm.reshape(len(Ytr_mm), -1).sum(axis=1))
pos = np.flatnonzero(flat_sums >= 10).astype(np.int64)
neg = np.flatnonzero(flat_sums < 10).astype(np.int64)

np.save(SEG_POS, pos)
np.save(SEG_NEG, neg)

Yva_mm = np.load(SEG_YVA, mmap_mode="r")
print(
    f"crops: train {len(Ytr_mm)} (with caries {len(pos)}, without {len(neg)}) "
    f"| val {len(Yva_mm)}"
)
assert len(pos) > 0, "No caries crops in the training split - masks/boxes are not aligned."

del Ytr_mm, Yva_mm, flat_sums, pos, neg
gc.collect()
torch.cuda.empty_cache()


seg_meta = {
    "xtr": str(SEG_XTR),
    "ytr": str(SEG_YTR),
    "xva": str(SEG_XVA),
    "yva": str(SEG_YVA),
    "pos": str(SEG_POS),
    "neg": str(SEG_NEG),
    "cfg": {
        "epochs": int(CFG["seg_epochs"]),
        "batch_per_gpu": 32,
        "lr": float(CFG["seg_lr"]),
        "patience": int(CFG["seg_patience"]),
        "budget_h": float(CFG["seg_budget_h"]),
        "seed": int(CFG["seed"]),
        "neg_ratio": float(CFG["seg_neg_ratio"]),
        "tv_alpha": float(CFG["seg_tv_alpha"]),
        "tv_beta": float(CFG["seg_tv_beta"]),
        "fp_lambda": float(CFG["seg_fp_lambda"]),
        "hn_start": int(CFG["seg_hn_start"]),
        "hn_every": int(CFG["seg_hn_every"]),
        "hn_frac": float(CFG["seg_hn_frac"]),
        "hn_min_px": int(CFG["seg_hn_min_px"]),
        "sel_fp_weight": float(CFG["seg_sel_fp_weight"]),
    },
}
with open(SEG_META, "wb") as f:
    pickle.dump(seg_meta, f, protocol=4)


ddp_script = r"""
import os
import gc
import math
import time
import random
import pickle
import resource

import numpy as np
import torch
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from segmentation_models_pytorch import Unet


def seed_all(seed, rank):
    s = seed + rank
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)


def ram_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 ** 2)


def prep(x):
    mean = torch.tensor(
        (0.485, 0.456, 0.406),
        device=x.device,
        dtype=x.dtype,
    ).view(1, 3, 1, 1)

    std = torch.tensor(
        (0.229, 0.224, 0.225),
        device=x.device,
        dtype=x.dtype,
    ).view(1, 3, 1, 1)

    return (x.repeat(1, 3, 1, 1) - mean) / std


def gpu_aug(x, y):
    B = x.shape[0]
    d = x.device

    flip = torch.rand(B, 1, 1, 1, device=d) < 0.5
    x = torch.where(flip, x.flip(-1), x)
    y = torch.where(flip, y.flip(-1), y)

    ang = (torch.rand(B, device=d) * 2 - 1) * math.radians(12)
    sc = 1 + (torch.rand(B, device=d) * 2 - 1) * 0.12
    tx = (torch.rand(B, device=d) * 2 - 1) * 0.08
    ty = (torch.rand(B, device=d) * 2 - 1) * 0.08

    c = torch.cos(ang) / sc
    s = torch.sin(ang) / sc

    theta = torch.stack(
        [
            torch.stack([c, -s, tx], 1),
            torch.stack([s, c, ty], 1),
        ],
        1,
    )

    grid = F.affine_grid(theta, list(x.shape), align_corners=False)

    x = F.grid_sample(
        x,
        grid,
        mode="bilinear",
        padding_mode="border",
        align_corners=False,
    )

    y = (
        F.grid_sample(
            y,
            grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        ) > 0.5
    ).float()

    a = 1 + (
        torch.rand(B, 1, 1, 1, device=d) * 2 - 1
    ) * 0.2

    b = (
        torch.rand(B, 1, 1, 1, device=d) * 2 - 1
    ) * 0.1

    return (x * a + b).clamp(0, 1), y


def seg_loss(logits, y, alpha=0.5, beta=0.5, fp_lambda=0.5):
    # Tversky: alpha weights false positives, beta false negatives. 0.5/0.5 = plain Dice (no recall bias).
    bce = F.binary_cross_entropy_with_logits(logits, y)
    p = torch.sigmoid(logits)
    tp = (p * y).sum()
    fp = (p * (1 - y)).sum()
    fn = ((1 - p) * y).sum()
    tversky = (tp + 1) / (tp + alpha * fp + beta * fn + 1)
    loss = bce + (1 - tversky)
    # The batch-pooled Tversky term barely notices a small blob in a caries-free crop. Penalise the strongest
    # activation of every empty crop directly, so a single confident false blob always costs something.
    empty = y.flatten(1).sum(1) == 0
    if fp_lambda > 0 and bool(empty.any()):
        loss = loss + fp_lambda * p.flatten(1).amax(1)[empty].mean()
    return loss


class CropDataset(Dataset):
    def __init__(self, x_path, y_path, indices):
        self.X = np.load(x_path, mmap_mode="r")
        self.Y = np.load(y_path, mmap_mode="r")
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        x = np.array(self.X[j], dtype=np.uint8, copy=True)
        y = np.array(self.Y[j], dtype=np.uint8, copy=True)
        return x, y


def collate(batch):
    xs, ys = zip(*batch)
    return (
        torch.from_numpy(np.stack(xs)),
        torch.from_numpy(np.stack(ys)),
    )


@torch.no_grad()
def validate(model, x_path, y_path, device, batch_size=128, thr=0.5, fp_min_px=30):
    # returns (micro Dice, fraction of caries-free crops that contain a predicted blob >= fp_min_px)
    model.eval()

    X = np.load(x_path, mmap_mode="r")
    Y = np.load(y_path, mmap_mode="r")

    inter = 0
    p_sum = 0
    g_sum = 0
    n_empty = 0
    n_empty_fp = 0

    for i in range(0, len(X), batch_size):
        xb = torch.from_numpy(
            np.array(X[i:i + batch_size], copy=True)
        ).float().div_(255.0)

        yb = torch.from_numpy(
            np.array(Y[i:i + batch_size], copy=True)
        ).float()

        xb = xb.unsqueeze(1).to(device)
        yb = yb.unsqueeze(1).to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(prep(xb))

        pm = torch.sigmoid(logits.float()) > thr
        gm = yb > 0

        inter += int(torch.logical_and(pm, gm).sum().item())
        p_sum += int(pm.sum().item())
        g_sum += int(gm.sum().item())

        empty = gm.flatten(1).sum(1) < 10
        n_empty += int(empty.sum().item())
        n_empty_fp += int((empty & (pm.flatten(1).sum(1) >= fp_min_px)).sum().item())

    return 2 * inter / max(p_sum + g_sum, 1), n_empty_fp / max(n_empty, 1)


@torch.no_grad()
def mine_hard_negatives(model, x_path, neg, rank, device, min_px, batch_size=128):
    # Score every caries-free training crop with the current model (work split across both ranks) and return the
    # crop ids where it predicts >= min_px pixels: these are the model's own false positives.
    model.eval()
    X = np.load(x_path, mmap_mode="r")
    scores = torch.zeros(len(neg), device=device)
    mine = np.arange(rank, len(neg), 2)
    for s in range(0, len(mine), batch_size):
        sel = mine[s:s + batch_size]
        xb = torch.from_numpy(
            np.stack([np.array(X[int(neg[j])], copy=True) for j in sel])
        ).float().div_(255.0).unsqueeze(1).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            lg = model(prep(xb))
        area = (torch.sigmoid(lg.float()) > 0.5).flatten(1).sum(1).float()
        scores[torch.from_numpy(sel).to(device)] = area
    dist.all_reduce(scores, op=dist.ReduceOp.SUM)
    return neg[(scores >= min_px).cpu().numpy()]


def main():
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])

    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")

    dist.init_process_group(
        backend="nccl",
        init_method="env://",
    )

    with open(os.environ["SEG_META_PATH"], "rb") as f:
        meta = pickle.load(f)

    cfg = meta["cfg"]
    seed_all(cfg["seed"], rank)

    pos = np.load(meta["pos"])
    neg = np.load(meta["neg"])

    k_neg = min(len(neg), int(round(float(cfg["neg_ratio"]) * len(pos))))
    batch_per_gpu = int(cfg["batch_per_gpu"])
    global_n = len(pos) + k_neg
    global_n -= global_n % 2          # even, so both ranks always run the same number of steps (DDP deadlock guard)
    local_n = global_n // 2
    hard_pool = np.zeros(0, dtype=np.int64)
    steps = math.ceil(local_n / batch_per_gpu)

    model = Unet(
        encoder_name="efficientnet-b0",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    ).to(device)

    ddp = DDP(
        model,
        device_ids=[local_rank],
        output_device=local_rank,
        broadcast_buffers=False,
        # EfficientNet-B0/U-Net can contain parameters that are not exercised
        # on every forward path under this SMP/torchvision combination.
        # Enable unused-parameter detection so DDP can safely synchronize them.
        find_unused_parameters=True,
    )

    optimizer = torch.optim.AdamW(
        ddp.parameters(),
        lr=cfg["lr"],
        weight_decay=1e-4,
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=cfg["lr"],
        total_steps=cfg["epochs"] * steps,
        pct_start=0.1,
    )

    scaler = make_scaler()
    best = -1.0
    bad = 0
    t_stage = time.time()

    for epoch in range(cfg["epochs"]):
        rng = np.random.RandomState(cfg["seed"] + epoch)

        # Hard-negative mining: every `hn_every` epochs (after a warm-up) find the caries-free crops the current
        # model still fires on. Both ranks run this (all_reduce inside), so it must stay outside `if rank == 0`.
        if epoch >= cfg["hn_start"] and (epoch - cfg["hn_start"]) % cfg["hn_every"] == 0:
            hard_pool = mine_hard_negatives(ddp.module, meta["xtr"], neg, rank, device, cfg["hn_min_px"])
            ddp.train()
            if rank == 0:
                print(f"[seg-DDP] hard-negative pool: {len(hard_pool)}/{len(neg)} caries-free crops", flush=True)

        n_hard = min(len(hard_pool), int(round(cfg["hn_frac"] * k_neg)))
        hard_sel = rng.choice(hard_pool, n_hard, replace=False) if n_hard > 0 else np.zeros(0, dtype=np.int64)
        rest = np.setdiff1d(neg, hard_pool)
        rand_sel = rng.choice(rest, min(len(rest), k_neg - n_hard), replace=False)
        selected_neg = np.concatenate([hard_sel, rand_sel]).astype(np.int64)
        if len(selected_neg) < k_neg:     # top up from whatever is left so the epoch size (and OneCycle steps) stay fixed
            left = np.setdiff1d(neg, selected_neg)
            selected_neg = np.concatenate([selected_neg, rng.choice(left, k_neg - len(selected_neg), replace=False)])

        indices = np.concatenate([pos, selected_neg])
        rng.shuffle(indices)
        indices = indices[:global_n]

        # Identical global selection on both ranks; each rank receives its own slice.
        local_indices = indices[rank::2]

        ds = CropDataset(
            meta["xtr"],
            meta["ytr"],
            local_indices,
        )

        loader = DataLoader(
            ds,
            batch_size=batch_per_gpu,
            shuffle=False,
            num_workers=0,
            pin_memory=False,
            collate_fn=collate,
            drop_last=False,
        )

        ddp.train()

        t0 = time.time()
        total_loss = 0.0
        count = 0

        for x_cpu, y_cpu in loader:
            x = x_cpu.float().div_(255.0).unsqueeze(1).to(device)
            y = y_cpu.float().unsqueeze(1).to(device)

            x, y = gpu_aug(x, y)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                logits = ddp(prep(x))
                loss = seg_loss(logits.float(), y, cfg["tv_alpha"], cfg["tv_beta"], cfg["fp_lambda"])

            if not torch.isfinite(loss):
                scheduler.step()
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(ddp.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += float(loss.detach())
            count += 1

            del x_cpu, y_cpu, x, y, logits, loss

        dist.barrier()

        if rank == 0:
            dice, fp_rate = validate(
                ddp.module,
                meta["xva"],
                meta["yva"],
                device,
                batch_size=128,
            )
            score = dice - cfg["sel_fp_weight"] * fp_rate

            avg_loss = total_loss / max(count, 1)
            peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 3)

            if score > best:
                best = score
                bad = 0
                torch.save(
                    {
                        k: v.detach().cpu()
                        for k, v in ddp.module.state_dict().items()
                    },
                    "/kaggle/working/seg_best.pt",
                )
            else:
                bad += 1

            print(
                f"[seg-DDP] ep {epoch+1:3d}/{cfg['epochs']} "
                f"loss {avg_loss:.4f} | "
                f"val Dice(crop, thr .5) {dice:.4f} | "
                f"empty-crop FP rate {fp_rate:.3f} | "
                f"score {score:.4f} | "
                f"best {best:.4f} | "
                f"peak VRAM {peak_vram:.2f}G | "
                f"worker max-RAM {ram_gb():.2f}G | "
                f"{time.time()-t0:.0f}s",
                flush=True,
            )

            stop = (
                bad >= cfg["patience"]
                or (time.time()-t_stage)/3600 > cfg["budget_h"]
            )
        else:
            stop = False

        flag = torch.tensor(
            [1 if stop else 0],
            dtype=torch.int32,
            device=device,
        )

        dist.broadcast(flag, src=0)

        del loader, ds
        gc.collect()
        torch.cuda.empty_cache()

        if bool(flag.item()):
            break

    dist.barrier()

    if rank == 0:
        print("U-Net DDP complete.")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()
"""

SEG_SCRIPT.write_text(ddp_script, encoding="utf-8")
SEG_SCRIPT.chmod(0o755)

# Run the DDP training job or reuse an explicitly requested segmentation checkpoint.
if CFG.get("seg_reuse_ckpt", False) and (OUT / "seg_best.pt").exists():
    print("Reusing existing seg_best.pt.")
else:
    env = os.environ.copy()
    env["SEG_META_PATH"] = str(SEG_META)

    print("Launching U-Net with 2-GPU DDP...")
    result = subprocess.run(
        [
            "torchrun",
            "--standalone",
            "--nproc_per_node=2",
            str(SEG_SCRIPT),
        ],
        env=env,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"U-Net DDP exited with code {result.returncode}. "
            "Use the rank-specific traceback printed above."
        )

if not (OUT / "seg_best.pt").exists():
    raise FileNotFoundError("seg_best.pt was not produced.")

# Rebuild a portable single-GPU model for inference/export after DDP finishes.
unet, ENC_USED, got_pre_u = build_unet(
    "efficientnet-b0",
    pretrained=False,
)
unet = unet.to(dev)

seg_sd = torch.load(
    OUT / "seg_best.pt",
    map_location="cpu",
)
unet.load_state_dict(seg_sd)
unet.eval()

print(
    "Loaded U-Net checkpoint:",
    OUT / "seg_best.pt",
    "| device:",
    next(unet.parameters()).device,
)

# Memory-map only the validation crops needed for threshold tuning.
Xva = np.load(SEG_XVA, mmap_mode="r")
Yva = np.load(SEG_YVA, mmap_mode="r")


@torch.no_grad()
def seg_probs(model, X, bs=64, tta=True):
    """Run U-Net inference on a NumPy array or read-only memmap and return probabilities
    (tta=True averages logits with the horizontally flipped pass, exactly as DualStagePredictor does)."""
    model.eval()
    out = []

    for i in range(0, len(X), bs):
        xb_np = np.array(X[i:i + bs], dtype=np.uint8, copy=True)
        xb = (
            torch.from_numpy(xb_np)
            .float()
            .div_(255.0)
            .unsqueeze(1)
            .to(dev)
        )

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(prep(xb)).float()
            if tta:
                logits = (logits + torch.flip(model(prep(torch.flip(xb, dims=[-1]))).float(), dims=[-1])) / 2

        out.append(
            torch.sigmoid(logits.float())
            .squeeze(1)
            .cpu()
            .numpy()
        )

        del xb_np, xb, logits

    return np.concatenate(out, axis=0) if out else np.zeros(
        (0, S, S), dtype=np.float32
    )


def micro_dice(P, Y, thr):
    pm, gm = P > thr, Y > 0
    return float(
        2 * np.logical_and(pm, gm).sum()
        / max(pm.sum() + gm.sum(), 1)
    )

Pva = seg_probs(unet, Xva)
_gva = np.asarray(Yva) > 0
_empty = _gva.reshape(len(_gva), -1).sum(1) < 10
sweep_s = {}
for t in np.arange(0.30, 0.96, 0.05):
    pm = Pva > t
    d = float(2 * np.logical_and(pm, _gva).sum() / max(pm.sum() + _gva.sum(), 1))
    fpr = float((pm[_empty].reshape(int(_empty.sum()), -1).sum(1) >= 30).mean()) if _empty.any() else 0.0
    sweep_s[round(float(t), 2)] = (d, fpr, d - CFG["seg_sel_fp_weight"] * fpr)
SEG_THR = max(sweep_s, key=lambda k: sweep_s[k][2])      # starting point; refined at full-image level in section 4

print("val crops:", len(_gva), f"({int(_empty.sum())} caries-free)")
print("thr  : crop-Dice | empty-crop FP rate | score")
for t, (d, fpr, sc) in sweep_s.items():
    print(f"{t:.2f} : {d:.4f} | {fpr:.3f} | {sc:.4f}" + ("   <-- SEG_THR (start)" if t == SEG_THR else ""))

del Xva, Yva, Pva, _gva, _empty
gc.collect()
torch.cuda.empty_cache()


## 4. Full-pipeline evaluation on the held-out test split


In [ ]:
det_core = det_core.to(dev).eval()

pred = DualStagePredictor(
    unet.module if isinstance(unet, nn.DataParallel) else unet,
    det_core,
    size=S, pad=PAD, det_thr=DET_THR, unet_thr=SEG_THR, unet_encoder=ENC_USED,
    det_min_size=DET_MIN, det_max_size=DET_MAX, device=dev,
    unet_hi=None, min_area=0, own_box_margin=CFG["pp_own_box_margin"], merge="weighted")

# ---------------------------------------------------------------- lesion-level helpers
def _ncomp(m):
    n, lab = cv2.connectedComponents(m.astype(np.uint8), connectivity=8)
    return n - 1, lab

def lesion_counts(pm, gm):
    """(#pred blobs, #pred blobs touching GT, #GT blobs, #GT blobs touched by a prediction)."""
    n_p, lp = _ncomp(pm); n_g, lg = _ncomp(gm)
    tp_p = len(np.unique(lp[gm & (lp > 0)])); tp_g = len(np.unique(lg[pm & (lg > 0)]))
    return n_p, tp_p, n_g, tp_g

# ---------------------------------------------------------------- tune post-processing on the held-out Stage-2 val set
# Probabilities are computed once (own-box merge, GT/pseudo boxes) and cached; then (lo, hi, min_area) is swept.
# Objective = 0.5 * pixel Dice + 0.5 * lesion-level F1 (a blob is a TP if it touches GT, a GT blob is found if a blob touches it),
# so spurious blobs are punished even when they are small in pixels.
t0 = time.time()
val_cache = []
for r in SEG_VAL_RECS:
    prob, _, _ = pred.predict_prob(r["img"], boxes=r["boxes"])
    val_cache.append((prob.astype(np.float16), r["mask"].astype(bool)))
print(f"cached {len(val_cache)} val probability maps ({time.time() - t0:.0f}s)")

def score_pp(lo, hi, min_area):
    TP = FP = FN = 0; Np = TPp = Ng = TPg = 0
    for prob, gm in val_cache:
        pm = postprocess_prob(prob.astype(np.float32), lo, hi, min_area).astype(bool)
        TP += int((pm & gm).sum()); FP += int((pm & ~gm).sum()); FN += int((~pm & gm).sum())
        a, b, c, d = lesion_counts(pm, gm); Np += a; TPp += b; Ng += c; TPg += d
    dice = 2 * TP / max(2 * TP + FP + FN, 1)
    lp, lr = TPp / max(Np, 1), TPg / max(Ng, 1)
    lf1 = 2 * lp * lr / max(lp + lr, 1e-9)
    return dict(lo=lo, hi=hi, min_area=min_area, dice=dice, pix_prec=TP / max(TP + FP, 1), pix_rec=TP / max(TP + FN, 1),
                les_prec=lp, les_rec=lr, les_f1=lf1, obj=0.5 * dice + 0.5 * lf1)

grid = [(lo, hi, ma) for lo in (0.4, 0.5, 0.6, 0.7) for hi in (None, 0.8, 0.9) for ma in (0, 150, 400, 800)]
PP_SWEEP = [score_pp(*g) for g in grid]
LEGACY = score_pp(0.5, None, 0)
BEST = max(PP_SWEEP, key=lambda d: d["obj"])
fmt = lambda d: (f"lo {d['lo']:.2f} hi {d['hi'] if d['hi'] else '-':>4} area {d['min_area']:>4} | dice {d['dice']:.4f} "
                 f"pixP {d['pix_prec']:.3f} pixR {d['pix_rec']:.3f} | lesionP {d['les_prec']:.3f} lesionR {d['les_rec']:.3f} F1 {d['les_f1']:.3f} | obj {d['obj']:.4f}")
print("legacy-style post-processing:", fmt(LEGACY))
print("top 5 settings:")
for d in sorted(PP_SWEEP, key=lambda d: -d["obj"])[:5]: print("  ", fmt(d))
SEG_THR, SEG_HI, SEG_MIN_AREA = BEST["lo"], BEST["hi"], BEST["min_area"]
pred.unet_thr, pred.unet_hi, pred.min_area = SEG_THR, SEG_HI, SEG_MIN_AREA
print(f"-> using lo={SEG_THR} hi={SEG_HI} min_area={SEG_MIN_AREA}  ({time.time() - t0:.0f}s total)")
del val_cache; gc.collect()

# ---------------------------------------------------------------- evaluation
def eval_pipeline(rs, use_gt_boxes, fp_min_px=30):
    TP = FP = FN = 0; per, ip, ig = [], [], []
    Np = TPp = Ng = TPg = 0; t_tp = t_fp = t_tn = t_fn = 0
    for r in rs:
        out = pred.predict(r["img"], boxes=r["boxes"] if use_gt_boxes else None)
        pm, gm = out["mask"].astype(bool), r["mask"].astype(bool)
        tp = int((pm & gm).sum()); fp = int((pm & ~gm).sum()); fn = int((~pm & gm).sum())
        TP += tp; FP += fp; FN += fn
        if gm.any(): per.append(2 * tp / max(2 * tp + fp + fn, 1))
        ip.append(pm.sum() >= 50); ig.append(gm.any())
        a, b, c, d = lesion_counts(pm, gm); Np += a; TPp += b; Ng += c; TPg += d
        for x1, y1, x2, y2 in r["boxes"].astype(int):          # tooth-level: the unit a dentist would care about
            g = gm[y1:y2, x1:x2].sum() >= 10; p = pm[y1:y2, x1:x2].sum() >= fp_min_px
            t_tp += int(g and p); t_fn += int(g and not p); t_fp += int(p and not g); t_tn += int(not g and not p)
    ip, ig = np.array(ip), np.array(ig)
    return dict(dice=2 * TP / max(2 * TP + FP + FN, 1), iou=TP / max(TP + FP + FN, 1), precision=TP / max(TP + FP, 1),
                recall=TP / max(TP + FN, 1), mean_dice_per_caries_image=float(np.mean(per)) if per else None,
                lesion_precision=TPp / max(Np, 1), lesion_recall=TPg / max(Ng, 1),
                fp_blobs_per_image=(Np - TPp) / max(len(rs), 1), fp_pixels_per_image=FP / max(len(rs), 1),
                tooth_sensitivity=t_tp / max(t_tp + t_fn, 1), tooth_specificity=t_tn / max(t_tn + t_fp, 1),
                n_teeth_with_caries=t_tp + t_fn, n_teeth_without=t_tn + t_fp,
                image_sensitivity=float((ip & ig).sum() / max(ig.sum(), 1)),
                image_specificity=float((~ip & ~ig).sum() / max((~ig).sum(), 1)) if (~ig).any() else None,  # None: test has no caries-free images
                n_images=len(rs))

def eval_legacy(rs, use_gt_boxes):
    """Same weights, old inference behaviour (max-merge over padded crops, thr 0.5, no hysteresis / area filter)."""
    keep = (pred.merge, pred.unet_thr, pred.unet_hi, pred.min_area)
    pred.merge, pred.unet_thr, pred.unet_hi, pred.min_area = "max", 0.5, None, 0
    try: return eval_pipeline(rs, use_gt_boxes)
    finally: pred.merge, pred.unet_thr, pred.unet_hi, pred.min_area = keep

RESULTS = dict(
    tooth_detection_test=DET_TEST,
    caries_with_gt_boxes=eval_pipeline(test, True),        # stage 2 alone (oracle boxes)
    caries_end_to_end=eval_pipeline(test, False),          # full dual-stage pipeline
    caries_with_gt_boxes_legacy_inference=eval_legacy(test, True),
    caries_end_to_end_legacy_inference=eval_legacy(test, False),
    postproc=dict(best=BEST, legacy=LEGACY), n_stage2_val=len(SEG_VAL_RECS),
    box_coverage_of_caries=COVER, det_thr=DET_THR, seg_thr=SEG_THR, seg_hi=SEG_HI, seg_min_area=SEG_MIN_AREA, encoder=ENC_USED,
    n_train=len(train), n_val=len(val), n_test=len(test))
for k in ("caries_with_gt_boxes", "caries_with_gt_boxes_legacy_inference", "caries_end_to_end", "caries_end_to_end_legacy_inference"):
    print(k, {a: (round(b, 4) if isinstance(b, float) else b) for a, b in RESULTS[k].items()})


## 5. Export: pickle bundle, results, curves, qualitative figures, round-trip check


In [ ]:
def clean(o): return json.loads(json.dumps(o, default=lambda v: float(v) if isinstance(v, (np.floating, np.integer)) else str(v)))

extra = clean(dict(dataset="testpeaky/dental (box-annotated) + DC1000 pseudo-labeled panoramics + "
                              "DC1000 pre-cropped tiles for Stage 2",
                   cfg=CFG, results=RESULTS, box_mode=MODE,
                   split=dict(train=[r["name"] for r in train], val=[r["name"] for r in val], test=[r["name"] for r in test]),
                   n_stage2_train_total=len(SEG_TRAIN_RECS), n_stage2_val=len(SEG_VAL_RECS),
                   postproc=dict(lo=SEG_THR, hi=SEG_HI, min_area=SEG_MIN_AREA, merge="weighted", own_box_margin=CFG["pp_own_box_margin"]),
                   n_stage2_train_boxed=len(train),
                   n_stage2_train_pseudo=n_pseudo,
                   n_stage2_train_precrop=n_precrop,
                   torch=torch.__version__))
PKL = OUT / "dental_caries_dualstage.pkl"
pred.save_pickle(str(PKL), extra=extra)
json.dump(clean(dict(results=RESULTS, hist=HIST)), open(OUT / "results.json", "w"), indent=2)
print(f"saved {PKL} ({PKL.stat().st_size / 1e6:.1f} MB)")

# round-trip: reload from the pickle and make sure it reproduces the in-memory model
chk = DualStagePredictor.from_pickle(str(PKL), device=dev)
r0 = test[0]
a = pred.predict(r0["img"], boxes=r0["boxes"])["prob"]; b = chk.predict(r0["img"], boxes=r0["boxes"])["prob"]
print("round-trip max |diff| (GT boxes):", float(np.abs(a - b).max()))
assert np.abs(a - b).max() < 1e-3, "reloaded pickle does not reproduce the trained model!"
o1, o2 = pred.predict(r0["img"]), chk.predict(r0["img"])
print("round-trip detector boxes equal:", o1["boxes"].shape == o2["boxes"].shape and np.allclose(o1["boxes"], o2["boxes"], atol=1e-2))

# training curves
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(HIST["det_f1"], label="val F1 @IoU .5"); ax[0].set_title("Stage 1: tooth detector"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(HIST["seg_dice"], label="val Dice (crop)"); ax[1].set_title("Stage 2: caries U-Net"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / "training_curves.png", dpi=150); plt.close()

# qualitative overlays: green = ground truth, red = prediction, yellow = detected teeth
(OUT / "qualitative").mkdir(exist_ok=True)
shown = 0
for r in test:
    if not r["mask"].any(): continue
    o = pred.predict(r["img"]); vis = cv2.cvtColor(r["img"], cv2.COLOR_GRAY2BGR)
    for x1, y1, x2, y2 in o["boxes"].astype(int): cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), 1)
    for m, col in ((r["mask"], (0, 255, 0)), (o["mask"], (0, 0, 255))):
        cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis, cs, -1, col, 2)
    cv2.imwrite(str(OUT / "qualitative" / f"{Path(r['name']).stem}.jpg"), vis, [cv2.IMWRITE_JPEG_QUALITY, 90])
    shown += 1
    if shown >= 8: break

print("\nFILES IN /kaggle/working:")
for p in sorted(OUT.glob("*")): print(f"  {p.name:32s} {(p.stat().st_size / 1e6 if p.is_file() else 0):8.1f} MB")
print(f"\nDONE in {elapsed_h():.2f} h.  Use it anywhere with:\n  from dental_inference import DualStagePredictor\n  pred = DualStagePredictor.from_pickle('dental_caries_dualstage.pkl'); out = pred.predict('img.png')")
